## Load Preprocessed Data

In [2]:
import numpy as np

class CustomLinearSVM:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):

        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters        
        self.w = None
        self.b = None

    def fit(self, X, y):
        
        n_samples, n_features = X.shape
        
        self.w = np.zeros(n_features)
        self.b = 0.0
        
        y_mirrored = np.where(y <= 0, -1, 1)

        for epoch in range(self.n_iters):
            
            for idx, x_i in enumerate(X):
                
                y_i = y_mirrored[idx]
                f_i= np.dot((self.w).T, x_i)+self.b

                if f_i * y_i <1:
                    self.w=self.w-self.lr*(self.lambda_param*self.w - y_i*x_i)
                    self.b = self.b + self.lr * y_i
                
                else:
                    self.w=self.w-self.lr*self.lambda_param*self.w 

                
              

    def predict(self, X):
       linear_output = np.dot(X, self.w) + self.b
       prediction = np.where(linear_output >= 0, 1, 0)
       return prediction
       


class OneVsRestSVM:
    def __init__(self, n_classes=10, learning_rate=0.001, lambda_param=0.01, n_iters=50):
        self.n_classes = n_classes
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.models = []

    def fit(self, X, y):
        for i in range(self.n_classes):
            print(f"    Training binary model for Class {i} vs Rest...")
            
            y_binary = np.where(y == i, 1, 0)
            
            binary_model = CustomLinearSVM(
                learning_rate=self.lr, 
                lambda_param=self.lambda_param, 
                n_iters=self.n_iters
            )
            binary_model.fit(X, y_binary)
            self.models.append(binary_model)

    def predict(self, X):
        n_samples = X.shape[0]
        scores = np.zeros((n_samples, self.n_classes))
        
        for i, model in enumerate(self.models):
            raw_value = np.dot(X, model.w) + model.b
            scores[:, i] = raw_value
        
        winning_classes = np.argmax(scores, axis=1)
        return winning_classes

## Training

In [2]:
import time

from preprocessing2 import preprocess
from sklearn.metrics import classification_report, confusion_matrix

X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="hog", n_pca=50)

# ==========================================
# 4. TRAIN AND EVALUATE
# ==========================================
print("\nInitializing Custom 10-Class SVM (OvR)...")
ovr_svm = OneVsRestSVM(n_classes=10, learning_rate=0.001, lambda_param=0.01, n_iters=50)

print("Training all 10 models... (This might take a few minutes in Python)")
start_time = time.time()
ovr_svm.fit(X_train, y_train)
print(f"Full 10-Class Training completed in {(time.time() - start_time):.2f} seconds.")

print("\nEvaluating on Validation Data...")
val_preds = ovr_svm.predict(X_val)

print("\n" + "="*50)
print("  10-CLASS MODEL PERFORMANCE (HOG + PCA FEATURES)")
print("="*50)
target_names = [f"Digit {i}" for i in range(10)]
print(classification_report(y_val, val_preds, target_names=target_names))


print("\nValidation Confusion Matrix (10x10):")
print(confusion_matrix(y_val, val_preds))

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000

Initializing Custom 10-Class SVM (OvR)...
Training all 10 models... (This might take a few minutes in Python)
    Training binary model for Class 0 vs Rest...
    Training binary model for Class 1 vs Rest...
    Training binary model for Class 2 vs Rest...
    Training binary model for Class 3 vs Rest...
    Training binary model for Class 4 vs Rest...
    Training binary model for Class 5 vs Rest...
    Training binary model for Class 6 vs Rest...
    Training binary model for Class 7 vs Rest...
    Training binary model for Class 8 vs Rest...
    Training binary model for Class 9 vs Rest...
Full 10-Class Training completed in 169.91 seconds.

Evaluating on Validation Data...

  10-CLASS MODEL PERFORMANCE (HOG + PCA FEATURES)
              precision    recall  f1-score   support

     Digit 0       0.96      0.98      0.97       587
     Digit 1       0.97      0.98      0.98       630
     Digit 2       0.93

In [ ]:
import time
from sklearn.metrics import classification_report, confusion_matrix
from preprocessing2 import cnn 

# 1. Manually slice and extract CNN features
X_train_cnn, y_train_cnn, X_val_cnn, y_val_cnn, X_test_cnn, y_test_cnn = cnn(subset_limit=10000)

# 2. Train Custom OvR SVM
print("\nInitializing Custom 10-Class SVM (OvR)...")
ovr_cnn_svm = OneVsRestSVM(n_classes=10, learning_rate=0.001, lambda_param=0.0001, n_iters=500)

print("Training all 10 models on CNN features...")
start_time = time.time()
ovr_cnn_svm.fit(X_train_cnn, y_train_cnn)
print(f"Full 10-Class CNN Training completed in {(time.time() - start_time):.2f} seconds.")

# 3. Evaluate
print("\nEvaluating on Validation Data...")
val_preds_cnn = ovr_cnn_svm.predict(X_val_cnn)

print("\n" + "="*50)
print("  10-CLASS MODEL PERFORMANCE (CNN FEATURES)")
print("="*50)
target_names = [f"Digit {i}" for i in range(10)]
print(classification_report(y_val_cnn, val_preds_cnn, target_names=target_names))

print("\nValidation Confusion Matrix (10x10):")
print(confusion_matrix(y_val_cnn, val_preds_cnn))

Extracting CNN Features...
250/250 ━━━━━━━━━━━━━━━━━━━━ 27s 106ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 100ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 97ms/step

Initializing Custom 10-Class SVM (OvR)...
Training all 10 models on CNN features...
    Training binary model for Class 0 vs Rest...
    Training binary model for Class 1 vs Rest...
    Training binary model for Class 2 vs Rest...
    Training binary model for Class 3 vs Rest...
    Training binary model for Class 4 vs Rest...
    Training binary model for Class 5 vs Rest...
    Training binary model for Class 6 vs Rest...
    Training binary model for Class 7 vs Rest...
    Training binary model for Class 8 vs Rest...
    Training binary model for Class 9 vs Rest...
Full 10-Class CNN Training completed in 168.92 seconds.

Evaluating on Validation Data...

  10-CLASS MODEL PERFORMANCE (CNN FEATURES)
              precision    recall  f1-score   support

     Digit 0       0.96      0.97      0.96       206
     Digit 1       0.97   